In [0]:
df = spark.table("workspace.default.churn_features")
display(df.limit(10))

CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Index,Gender_Index,Normalized_Balance,Normalized_EstimatedSalary
15634602,619,France,Female,42,2,0.0,1,1,1,101348.88,1,0.0,1.0,0.0,0.5067348931822989
15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,2.0,1.0,0.334031478677259,0.5627087386845443
15619304,502,France,Female,42,8,159660.8,3,1,0,113931.57,1,0.0,1.0,0.6363571759354565,0.5696543519906151
15701354,699,France,Female,39,1,0.0,2,0,0,93826.63,0,0.0,1.0,0.0,0.4691200509648671
15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.1,0,2.0,1.0,0.5002462155052676,0.3954003607344501
15574012,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1,2.0,0.0,0.45339436422174445,0.7487971601287923
15592531,822,France,Male,50,7,0.0,2,1,1,10062.8,0,0.0,0.0,0.0,0.050260899915941966
15656148,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1,1.0,1.0,0.4585397202505608,0.5967334880481086
15792365,501,France,Male,44,4,142051.07,2,0,1,74940.5,0,0.0,0.0,0.5661703921301275,0.3746803819764787
15592389,684,France,Male,27,2,134603.88,1,1,1,71725.73,0,0.0,0.0,0.5364882610306042,0.35860499677719215


In [0]:
df.createOrReplaceTempView("churn_data")
print("Temporary SQL view created: churn_data")

Temporary SQL view created: churn_data


In [0]:
%sql
SELECT 
    IsActiveMember,
    COUNT(*) AS Total_Customers,
    SUM(Exited) AS Churned_Customers,
    SUM(Exited) * 1.0 / COUNT(*) AS Churn_Rate
FROM churn_data
GROUP BY IsActiveMember
ORDER BY IsActiveMember DESC

IsActiveMember,Total_Customers,Churned_Customers,Churn_Rate
1,5151,735,0.1426907396622015
0,4849,1302,0.2685089709218396


**Insight**
- Inactive users churn almost 2x more
- Strong behavioural signal

**Business action**
- target inactive users immediately
- re-engagement campaigns:
  - offers
  - remainders
  - product nudges

In [0]:
%sql
SELECT 
    Geography,
    COUNT(*) AS Total_Customers,
    SUM(Exited) AS Churned_Customers,
    SUM(Exited) * 1.0 / COUNT(*) AS Churn_Rate
FROM churn_data
GROUP BY Geography
ORDER BY Churn_Rate DESC

Geography,Total_Customers,Churned_Customers,Churn_Rate
Germany,2509,814,0.3244320446392985
Spain,2477,413,0.1667339523617279
France,5014,810,0.1615476665337056


**Insight**
- Germany has very high churn risk
- Almost double compared to others

**Business Action**
- Investigate Germany market:
  - pricing
  - competition
  - customer experience
- run region-specific retention strategy 

In [0]:
%sql
SELECT 
    NumOfProducts,
    COUNT(*) AS Total_Customers,
    SUM(Exited) AS Churned_Customers,
    SUM(Exited) * 1.0 / COUNT(*) AS Churn_Rate
FROM churn_data
GROUP BY NumOfProducts
ORDER BY NumOfProducts

NumOfProducts,Total_Customers,Churned_Customers,Churn_Rate
1,5084,1409,0.2771439811172305
2,4590,348,0.0758169934640523
3,266,220,0.8270676691729323
4,60,60,1.0000000000000000


**Insight**
- Customers with 2 products are most stable
- 1-product customers are high risk

**Business Action**
- push cross-selling:
  - move customers from 1 → 2 products
- bundle products

In [0]:
%sql
SELECT 
    SUM(Balance) AS Total_Balance_At_Risk
FROM churn_data
WHERE Exited = 1

Total_Balance_At_Risk
1.855880946299999E8


**Insight**
- churn is not just customers loss
- it is massive financial risk

**Business Action**
- justify retention budget
- prioritize high-balance customers

In [0]:
%sql
SELECT 
    CustomerId,
    Balance,
    IsActiveMember
FROM churn_data
WHERE IsActiveMember = 0
  AND Balance > 100000
ORDER BY Balance DESC
LIMIT 100

CustomerId,Balance,IsActiveMember
15714241,222267.63,0
15571958,221532.8,0
15599131,214346.96,0
15594408,213146.2,0
15620268,212696.32,0
15780212,212692.97,0
15671256,211774.31,0
15709920,208165.53,0
15769412,207034.96,0
15627971,206663.75,0


**Insight**
- This is the targetable customer list
- these are immediate intervention candidates

**Business Actions**
- send list to:
    - CRM team
    - sales team
- offer:
    - discounts
    - premium services